In [ ]:
import os
import json
import zipfile
import random
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# --- STEP 1: KAGGLE API CONFIGURATION ---
# Setting up credentials to download the chest X-ray dataset automatically
os.environ['KAGGLE_USERNAME'] = "kumarnemade"
os.environ['KAGGLE_KEY'] = "ea2ff6ea7bbb72f283573ae559aa6506"

!mkdir -p ~/.kaggle
data = {"username":"kumarnemade", "key":"ea2ff6ea7bbb72f283573ae559aa6506"}
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(data, f)
!chmod 600 ~/.kaggle/kaggle.json

print("API Key configured successfully!")

# --- STEP 2: DOWNLOAD & UNZIP DATASET ---
# Fetching the Pneumonia dataset directly from Kaggle
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

print("Unzipping images... please wait.")
with zipfile.ZipFile('chest-xray-pneumonia.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset_folder')
print("All 5,000+ images are ready in 'dataset_folder'!")

# --- STEP 3: LOAD YOUR TRAINED AI MODEL ---
# Loading the CNN model created in your previous project
# Ensure 'gtu_pneumonia_model.h5' is uploaded to your Colab session
if os.path.exists('gtu_pneumonia_model.h5'):
    model = tf.keras.models.load_model('gtu_pneumonia_model.h5')
    print("Model loaded successfully!")
else:
    print("Error: Please upload 'gtu_pneumonia_model.h5' to the Colab folder.")

# --- STEP 4: DATA PREPARATION FOR 50 PATIENTS ---
# Loading your structured health indicators
df_health = pd.read_csv('diabetes_012_health_indicators_BRFSS2015.csv')

# Defining paths for random image selection
test_path_p = 'dataset_folder/chest_xray/test/PNEUMONIA/'
test_path_n = 'dataset_folder/chest_xray/test/NORMAL/'
all_images = [os.path.join(test_path_p, f) for f in os.listdir(test_path_p)] + \
             [os.path.join(test_path_n, f) for f in os.listdir(test_path_n)]

# --- STEP 5: GENERATE MULTI-PATIENT ECOSYSTEM DATA ---
# We are creating 50 rows so your Power BI Dashboard looks professional.
results_list = []

print("Processing 50 patient records... Integrating Image AI with Clinical Data.")

for i in range(50):
    # Select random image and random patient clinical history
    img_path = random.choice(all_images)
    random_patient = df_health.sample(n=1).iloc[0]

    # 1. Image Diagnosis (CNN Layer)
    img = image.load_img(img_path, target_size=(150, 150))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    ai_diagnosis = 1 if prediction[0] > 0.5 else 0

    # 2. Intelligent Triage Logic (Decision Layer)
    # Check for underlying health indicators: High BP or Diabetes
    has_risks = (random_patient['HighBP'] == 1 or random_patient['Diabetes_012'] > 0)

    if ai_diagnosis == 1 and has_risks:
        priority = "CRITICAL"
        action = "Immediate ICU Admission"
    elif ai_diagnosis == 1:
        priority = "HIGH"
        action = "General Ward Monitoring"
    else:
        priority = "STABLE"
        action = "Outpatient/Home Care"

    # Append data for CSV export
    results_list.append({
        'Patient_ID': f"P-{1000 + i}",
        'Diagnosis': 'Pneumonia' if ai_diagnosis == 1 else 'Normal',
        'High_BP': 'Yes' if random_patient['HighBP'] == 1 else 'No',
        'Diabetes': 'Yes' if random_patient['Diabetes_012'] > 0 else 'No',
        'Priority_Status': priority,
        'Hospital_Action': action
    })

# --- STEP 6: EXPORT TO POWER BI ---
final_report = pd.DataFrame(results_list)
final_report.to_csv('hospital_operations_data.csv', index=False)

print(f"\nSUCCESS: 'hospital_operations_data.csv' generated with {len(final_report)} rows.")
print(final_report.head())

API Key configured successfully!
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:15<00:00, 157MB/s] 

Unzipping images... please wait.
All 5,000+ images are ready in 'dataset_folder'!


Model loaded successfully!
Processing 50 patient records... Integrating Image AI with Clinical Data.

SUCCESS: 'hospital_operations_data.csv' generated with 50 rows.
  Patient_ID  Diagnosis High_BP Diabetes Priority_Status  \
0     P-1000  Pneumonia     Yes      Yes        CRITICAL   
1     P-1001  Pneumonia      No       No            HIGH   
2     P-1002  Pneumonia      No      Yes        CRITICAL   
3     P-1003  Pneumonia      No       No            HIGH   
4     P-1004  Pneumonia     Yes       No        CRITICAL   

           Hospital_Action  
0  Immediate ICU Admission  
1  General Ward Monitoring  
2  Immediate ICU Admission  
3  General Ward Monitoring  
4  Immediate ICU Admission  
